<a href="https://colab.research.google.com/github/danilo576/Cuda/blob/main/CUDA_Paralelni_sistemi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUDA — Paralelni sistemi

Materijal i vežbe iz programiranja na GPU-u pomoću NVIDIA CUDA tehnologije.

# ⚙️ 1. Podešavanje okruženja

In [14]:
!pip install -q nvcc4jupyter

In [15]:
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


## 1.1 Provera GPU-a

## 1.2 Provera CUDA kompajlera

In [16]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


# 🚀 2. Prvi CUDA program

Pre nego što krenemo sa vežbama, napravićemo jednostavan CUDA program koji izvršava funkciju na GPU-u.

In [17]:
%%cuda

#include <stdio.h>

__global__ void hello() {
    printf("Hello from GPU!\n");
}

int main() {
    hello<<<1, 1>>>();
    cudaDeviceSynchronize();

    return 0;
}

Hello from GPU!



## 2.1 CPU i GPU

In [18]:
%%cuda

#include <stdio.h>

__global__ void hello_from_gpu() {
    printf("Hello from GPU!\n");
}

int main() {

    printf("Hello from CPU!\n");

    hello_from_gpu<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

Hello from CPU!
Hello from GPU!



## 2.2 Blokovi i threadovi

In [19]:
%%cuda

#include <stdio.h>

__global__ void hello() {

    printf(
        "Block: %d, Thread: %d\n",
        blockIdx.x,
        threadIdx.x
    );
}

int main() {

    hello<<<2, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}

Block: 1, Thread: 0
Block: 1, Thread: 1
Block: 1, Thread: 2
Block: 1, Thread: 3
Block: 0, Thread: 0
Block: 0, Thread: 1
Block: 0, Thread: 2
Block: 0, Thread: 3



# 🧮 3. Sabiranje vektora

Sabiranje dva vektora korišćenjem CUDA kernela.

Za svaki element važi:
$
C[i] = A[i] + B[i]
$

In [20]:
%%cuda

#include <stdio.h>

#define N 10

__global__ void vector_add(const int *A, const int *B, int *C) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {

    int A[N], B[N], C[N];

    for (int i = 0; i < N; i++) {
        A[i] = i;
        B[i] = i * 2;
    }

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, N * sizeof(int));
    cudaMalloc(&d_B, N * sizeof(int));
    cudaMalloc(&d_C, N * sizeof(int));

    cudaMemcpy(d_A, A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N * sizeof(int), cudaMemcpyHostToDevice);

    vector_add<<<1, N>>>(d_A, d_B, d_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, N * sizeof(int), cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++) {
        printf("C[%d] = %d\n", i, C[i]);
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

C[0] = 0
C[1] = 3
C[2] = 6
C[3] = 9
C[4] = 12
C[5] = 15
C[6] = 18
C[7] = 21
C[8] = 24
C[9] = 27



# 📝 Zadatak 1 — Operacija nad vektorima

Napisati CUDA program koji računa izraz:

$
C = A * x + B
$

gde su:

- `A` i `B` — vektori,
- `x` — skalar,
- `C` — rezultujući vektor.

Za svaki element važi:

$
C[i] = A[i] * x + B[i]
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti rezultate i proveriti validnost CUDA rešenja.
4. Analizirati ponašanje programa pri promeni:
   - broja blokova,
   - broja niti u jednom bloku.

U CUDA kernelu koristi se globalni indeks niti:

```cpp
int tid = blockIdx.x * blockDim.x + threadIdx.x;

In [21]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 5120
#define BLOCK_SIZE 32

__host__ __device__ int operation(int a, int b, int x)
{
    return a * x + b;
}

__global__ void operate(int *A, int *B, int *C, int x)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    for (int i = tid; i < N; i += gridDim.x * blockDim.x)
    {
        C[i] = operation(A[i], B[i], x);
    }
}

__host__ bool check_result(int *A, int *B, int *GPU_C, int x)
{
    for (int i = 0; i < N; i++)
    {
        int CPU_C = operation(A[i], B[i], x);

        if (GPU_C[i] != CPU_C)
        {
            return false;
        }
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C, int x)
{
    int *dev_A;
    int *dev_B;
    int *dev_C;

    cudaMalloc((void **)&dev_A, N * sizeof(int));
    cudaMalloc((void **)&dev_B, N * sizeof(int));
    cudaMalloc((void **)&dev_C, N * sizeof(int));

    cudaMemcpy(dev_A, A, N * sizeof(int), cudaMemcpyHostToDevice);

    cudaMemcpy(
        dev_B,
        B,
        N * sizeof(int),
        cudaMemcpyHostToDevice
    );

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((N + BLOCK_SIZE - 1) / BLOCK_SIZE);

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C, x);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, N * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

int main()
{
    int *A;
    int *B;
    int *GPU_C;

    int x = 5;

    A = (int *)malloc(N * sizeof(int));
    B = (int *)malloc(N * sizeof(int));
    GPU_C = (int *)malloc(N * sizeof(int));

    for (int i = 0; i < N; i++)
    {
        A[i] = i + 1;
        B[i] = N - (i + 1);
    }

    operate_on_GPU(A, B, GPU_C, x);

    printf("Prvih 10 elemenata vektora C:\n\n");

    for (int i = 0; i < 10; i++)
    {
        printf("C[%d] = %d\n", i, GPU_C[i]);
    }

    if (check_result(A, B, GPU_C, x))
    {
        printf("\nRezultat je tacan!\n");
    }
    else
    {
        printf("\nRezultat je netacan!\n");
    }

    free(A);
    free(B);
    free(GPU_C);

    return 0;
}

Prvih 10 elemenata vektora C:

C[0] = 5124
C[1] = 5128
C[2] = 5132
C[3] = 5136
C[4] = 5140
C[5] = 5144
C[6] = 5148
C[7] = 5152
C[8] = 5156
C[9] = 5160

Rezultat je tacan!



# 📝 Zadatak 2 — Sabiranje dve matrice

Napisati CUDA program koji računa izraz:

$
C = A + B
$

gde su:

- `A` i `B` — kvadratne matrice dimenzija `N × N`,
- `C` — rezultujuća matrica.

Za svaki element važi:

$
C[i][j] = A[i][j] + B[i][j]
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti rezultate i proveriti validnost CUDA rešenja.
4. Analizirati ponašanje programa pri promeni:
   - broja blokova,
   - broja niti u okviru jednog bloka.

Pošto obrađujemo matricu, koristimo dvodimenzionalne blokove i grid.

Globalni indeks vrste i kolone:

```cpp
int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
int tid_y = blockIdx.y * blockDim.y + threadIdx.y;


In [22]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 512
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C);
__global__ void operate(int *A, int *B, int *C);
__host__ bool check_result(int *A, int *B, int *GPU_C);

__device__ __host__ int operation(int a, int b)
{
    return a + b;
}

int main()
{
    int *A, *B, *GPU_C;

    A = (int *)malloc(sizeof(int) * N * N);
    B = (int *)malloc(sizeof(int) * N * N);
    GPU_C = (int *)malloc(sizeof(int) * N * N);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            A[i * N + j] = i * N + j + 1;
            B[i * N + j] = j * N + i + 1;
        }
    }

    operate_on_GPU(A, B, GPU_C);

    printf("Prvih 8x8 elemenata matrice C:\n\n");

    for (int i = 0; i < 8; i++)
    {
        printf("|\t");

        for (int j = 0; j < 8; j++)
            printf("%d\t", GPU_C[i * N + j]);

        printf("|\n");
    }

    if (check_result(A, B, GPU_C))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat je netacan!\n");

    free(GPU_C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *GPU_C)
{
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            int index = i * N + j;

            if (GPU_C[index] != operation(A[index], B[index]))
                return false;
        }
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE);

    dim3 gridSize(
        (N + blockSize.x - 1) / blockSize.x,
        (N + blockSize.y - 1) / blockSize.y
    );

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C)
{
    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int tid_y = blockIdx.y * blockDim.y + threadIdx.y;

    for (int i = tid_x; i < N; i += gridDim.x * blockDim.x)
    {
        for (int j = tid_y; j < N; j += gridDim.y * blockDim.y)
        {
            C[i * N + j] = operation(A[i * N + j], B[i * N + j]);
        }
    }
}

Prvih 8x8 elemenata matrice C:

|	2	515	1028	1541	2054	2567	3080	3593	|
|	515	1028	1541	2054	2567	3080	3593	4106	|
|	1028	1541	2054	2567	3080	3593	4106	4619	|
|	1541	2054	2567	3080	3593	4106	4619	5132	|
|	2054	2567	3080	3593	4106	4619	5132	5645	|
|	2567	3080	3593	4106	4619	5132	5645	6158	|
|	3080	3593	4106	4619	5132	5645	6158	6671	|
|	3593	4106	4619	5132	5645	6158	6671	7184	|

Rezultat je tacan!



# 📝 Zadatak 3 — Sabiranje vektora sa zamenom elemenata

Napisati CUDA program koji računa izraz:

$
C = A + B
$

gde su:

- `A` i `B` — vektori,
- `C` — rezultujući vektor,
- `k` — granična vrednost,
- `p` — vrednost kojom se vrši zamena.

Za svaki element najpre računamo:

$
C[i] = A[i] + B[i]
$

Ako je zbir veći od `k`, rezultat se zamenjuje vrednošću `p`:

$
C[i] =
\begin{cases}
p, & \text{ako } A[i] + B[i] > k \\
A[i] + B[i], & \text{ako } A[i] + B[i] \leq k
\end{cases}
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Omogućiti unos vrednosti `k` i `p`.
3. Izračunati isti rezultat sekvencijalno na CPU-u.
4. Uporediti CPU i GPU rezultat.
5. Analizirati ponašanje programa pri promeni broja blokova i broja niti po bloku.

Globalni indeks niti računa se kao:

```cpp
int tid = blockIdx.x * blockDim.x + threadIdx.x;
```

Za obradu vektora koristi se grid-stride petlja:

```cpp
for (int i = tid; i < N; i += gridDim.x * blockDim.x)
```

In [23]:
%%writefile zadatak5.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 5120
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C, int k, int p);
__global__ void operate(int *A, int *B, int *C, int k, int p);
__host__ bool check_result(int *A, int *B, int *GPU_C, int k, int p);

__device__ __host__ int operation(int a, int b, int k, int p)
{
    int result = a + b;
    return result > k ? p : result;
}

int main(int argc, char **argv)
{
    int *A, *B, *GPU_C;
    int k, p;

    if (argc == 3)
    {
        k = atoi(argv[1]);
        p = atoi(argv[2]);
    }
    else
    {
        k = 17090;
        p = 2904;
    }

    A = (int *)malloc(sizeof(int) * N);
    B = (int *)malloc(sizeof(int) * N);
    GPU_C = (int *)malloc(sizeof(int) * N);

    for (int i = 0; i < N; i++)
    {
        A[i] = i * i;
        B[i] = N - (i + 1);
    }

    operate_on_GPU(A, B, GPU_C, k, p);

    printf("k = %d, p = %d\n\n", k, p);
    printf("Prvih 20 elemenata vektora C:\n");

    for (int i = 0; i < 20; i++)
        printf("C[%d] = %d\n", i, GPU_C[i]);

    if (check_result(A, B, GPU_C, k, p))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat je netacan!\n");

    free(GPU_C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *GPU_C, int k, int p)
{
    for (int i = 0; i < N; i++)
    {
        if (GPU_C[i] != operation(A[i], B[i], k, p))
            return false;
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C, int k, int p)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((N + blockSize.x - 1) / blockSize.x);

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C, k, p);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C, int k, int p)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    for (int i = tid; i < N; i += gridDim.x * blockDim.x)
        C[i] = operation(A[i], B[i], k, p);
}

Writing zadatak5.cu


In [24]:
!nvcc zadatak5.cu -o zadatak5

In [25]:
k = 5200 #@param {type:"integer"}
p = 100 #@param {type:"integer"}

!./zadatak5 {k} {p}

k = 5200, p = 100

Prvih 20 elemenata vektora C:
C[0] = 5119
C[1] = 5119
C[2] = 5121
C[3] = 5125
C[4] = 5131
C[5] = 5139
C[6] = 5149
C[7] = 5161
C[8] = 5175
C[9] = 5191
C[10] = 100
C[11] = 100
C[12] = 100
C[13] = 100
C[14] = 100
C[15] = 100
C[16] = 100
C[17] = 100
C[18] = 100
C[19] = 100

Rezultat je tacan!


# 📝 Zadatak 4 — Sabiranje matrica sa zamenom elemenata

Napisati CUDA program koji računa izraz:

$
C = A + B
$

gde su:

- `A` i `B` — kvadratne matrice dimenzija `N × N`,
- `C` — rezultujuća matrica,
- `k` — granična vrednost,
- `p` — vrednost kojom se vrši zamena.

Za svaki element najpre računamo:

$
C[i][j] = A[i][j] + B[i][j]
$

Ako je dobijena vrednost veća od `k`, element se zamenjuje vrednošću `p`:

$
C[i][j] =
\begin{cases}
p, & \text{ako } A[i][j] + B[i][j] > k \\
A[i][j] + B[i][j], & \text{ako } A[i][j] + B[i][j] \leq k
\end{cases}
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Omogućiti unos vrednosti `k` i `p`.
3. Izračunati isti rezultat sekvencijalno na CPU-u.
4. Uporediti CPU i GPU rezultat.
5. Analizirati ponašanje programa pri promeni broja blokova i broja niti po bloku.

Za matricu koristimo dvodimenzionalnu organizaciju niti:

```cpp
int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
int tid_y = blockIdx.y * blockDim.y + threadIdx.y;
```

Za obradu matrice koristimo grid-stride petlje.

In [26]:
%%writefile zadatak7.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 512
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C, int k, int p);
__global__ void operate(int *A, int *B, int *C, int k, int p);
__host__ bool check_result(int *A, int *B, int *C, int k, int p);

__device__ __host__ int operation(int a, int b, int k, int p)
{
    int result = a + b;
    return result > k ? p : result;
}

int main(int argc, char **argv)
{
    int *A, *B, *C;
    int k, p;

    A = (int *)malloc(sizeof(int) * N * N);
    B = (int *)malloc(sizeof(int) * N * N);
    C = (int *)malloc(sizeof(int) * N * N);

    if (argc == 3)
    {
        k = atoi(argv[1]);
        p = atoi(argv[2]);
    }
    else
    {
        k = 17090;
        p = 2904;
    }

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            A[i * N + j] = i * N + j;
            B[i * N + j] = j * N + i;
        }
    }

    operate_on_GPU(A, B, C, k, p);

    printf("k = %d, p = %d\n\n", k, p);
    printf("Prvih 8x8 elemenata matrice C:\n\n");

    for (int i = 0; i < 8; i++)
    {
        printf("|\t");

        for (int j = 0; j < 8; j++)
            printf("%d\t", C[i * N + j]);

        printf("|\n");
    }

    if (check_result(A, B, C, k, p))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat nije tacan!\n");

    free(C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *C, int k, int p)
{
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            int index = i * N + j;

            if (C[index] != operation(A[index], B[index], k, p))
                return false;
        }
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C, int k, int p)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridSize(
        (N + blockSize.x - 1) / blockSize.x,
        (N + blockSize.y - 1) / blockSize.y
    );

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C, k, p);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C, int k, int p)
{
    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int tid_y = blockIdx.y * blockDim.y + threadIdx.y;

    for (int i = tid_x; i < N; i += gridDim.x * blockDim.x)
    {
        for (int j = tid_y; j < N; j += gridDim.y * blockDim.y)
        {
            int index = i * N + j;
            C[index] = operation(A[index], B[index], k, p);
        }
    }
}

Writing zadatak7.cu


In [27]:
!nvcc zadatak7.cu -o zadatak7

In [28]:
k = 5000 #@param {type:"integer"}
p = 8 #@param {type:"integer"}

!./zadatak7 {k} {p}

k = 5000, p = 8

Prvih 8x8 elemenata matrice C:

|	0	513	1026	1539	2052	2565	3078	3591	|
|	513	1026	1539	2052	2565	3078	3591	4104	|
|	1026	1539	2052	2565	3078	3591	4104	4617	|
|	1539	2052	2565	3078	3591	4104	4617	8	|
|	2052	2565	3078	3591	4104	4617	8	8	|
|	2565	3078	3591	4104	4617	8	8	8	|
|	3078	3591	4104	4617	8	8	8	8	|
|	3591	4104	4617	8	8	8	8	8	|

Rezultat je tacan!


# 📝 Zadatak 5 — Množenje matrice skalarom

Napisati CUDA program koji računa izraz:

$
C = A \cdot x
$

gde su:

- `A` — kvadratna matrica dimenzija `N × N`,
- `x` — skalar,
- `C` — rezultujuća matrica.

Za svaki element važi:

$
C[i][j] = A[i][j] \cdot x
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti CPU i GPU rezultat.
4. Analizirati ponašanje programa pri promeni broja blokova i broja niti po bloku.

Za matricu koristimo dvodimenzionalnu organizaciju niti:

```cpp
int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
int tid_y = blockIdx.y * blockDim.y + threadIdx.y;
```

Za obradu matrice koriste se grid-stride petlje.

In [29]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 512
#define BLOCK_SIZE 32
#define X 5

__host__ void operate_on_GPU(int *A, int *C);
__global__ void operate(int *A, int *C);
__host__ bool check_result(int *A, int *C);

__device__ __host__ int operation(int a)
{
    return a * X;
}

int main()
{
    int *A, *C;

    A = (int *)malloc(sizeof(int) * N * N);
    C = (int *)malloc(sizeof(int) * N * N);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
            A[i * N + j] = i * N + j + 1;
    }

    operate_on_GPU(A, C);

    printf("x = %d\n\n", X);
    printf("Prvih 8x8 elemenata matrice C:\n\n");

    for (int i = 0; i < 8; i++)
    {
        printf("|\t");

        for (int j = 0; j < 8; j++)
            printf("%d\t", C[i * N + j]);

        printf("|\n");
    }

    if (check_result(A, C))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat nije tacan!\n");

    free(C);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *C)
{
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            int index = i * N + j;

            if (C[index] != operation(A[index]))
                return false;
        }
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *C)
{
    int *dev_A, *dev_C;
    size_t size = sizeof(int) * N * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridSize(
        (N + blockSize.x - 1) / blockSize.x,
        (N + blockSize.y - 1) / blockSize.y
    );

    operate<<<gridSize, blockSize>>>(dev_A, dev_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *C)
{
    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int tid_y = blockIdx.y * blockDim.y + threadIdx.y;

    for (int i = tid_x; i < N; i += gridDim.x * blockDim.x)
    {
        for (int j = tid_y; j < N; j += gridDim.y * blockDim.y)
        {
            int index = i * N + j;
            C[index] = operation(A[index]);
        }
    }
}

x = 5

Prvih 8x8 elemenata matrice C:

|	5	10	15	20	25	30	35	40	|
|	2565	2570	2575	2580	2585	2590	2595	2600	|
|	5125	5130	5135	5140	5145	5150	5155	5160	|
|	7685	7690	7695	7700	7705	7710	7715	7720	|
|	10245	10250	10255	10260	10265	10270	10275	10280	|
|	12805	12810	12815	12820	12825	12830	12835	12840	|
|	15365	15370	15375	15380	15385	15390	15395	15400	|
|	17925	17930	17935	17940	17945	17950	17955	17960	|

Rezultat je tacan!



# 📝 Zadatak 6 — Brojanje pojavljivanja elementa u matrici

Napisati CUDA program koji određuje koliko puta se element `k` pojavljuje u kvadratnoj matrici `A`.

Za svaki element matrice proveravamo:

$
A[i][j] = k
$

Ako je uslov ispunjen, brojač se uvećava za 1.

Potrebno je:

1. Izbrojati pojavljivanja elementa `k` paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti CPU i GPU rezultat.
4. Analizirati ponašanje programa pri promeni broja blokova i broja niti po bloku.

Pošto više niti može istovremeno da uvećava isti brojač, koristi se:

```cpp
atomicAdd(count, 1);
```

Za matricu koristimo dvodimenzionalnu organizaciju niti:

```cpp
int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
int tid_y = blockIdx.y * blockDim.y + threadIdx.y;
```

Za obradu matrice koriste se grid-stride petlje.

In [30]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 4096
#define BLOCK_SIZE 32
#define K 5
#define ITERATIONS 100

__global__ void operate_old(int *A, int *count);
__global__ void operate_optimized(int *A, int *count);
__host__ int check_result(int *A);

int main()
{
    int *A;
    int *dev_A, *dev_count;
    int CPU_count;
    int old_count;
    int optimized_count;

    A = (int *)malloc(sizeof(int) * N * N);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
            A[i * N + j] = (i * N + j) % 10;
    }

    CPU_count = check_result(A);

    size_t size = sizeof(int) * N * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_count, sizeof(int));

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridSize(
        (N + blockSize.x - 1) / blockSize.x,
        (N + blockSize.y - 1) / blockSize.y
    );

    cudaMemset(dev_count, 0, sizeof(int));

    operate_old<<<gridSize, blockSize>>>(dev_A, dev_count);
    cudaDeviceSynchronize();

    cudaMemcpy(&old_count, dev_count, sizeof(int), cudaMemcpyDeviceToHost);

    cudaMemset(dev_count, 0, sizeof(int));

    operate_optimized<<<gridSize, blockSize>>>(dev_A, dev_count);
    cudaDeviceSynchronize();

    cudaMemcpy(&optimized_count, dev_count, sizeof(int), cudaMemcpyDeviceToHost);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    float old_time;
    float optimized_time;

    cudaMemset(dev_count, 0, sizeof(int));

    cudaEventRecord(start);

    for (int i = 0; i < ITERATIONS; i++)
        operate_old<<<gridSize, blockSize>>>(dev_A, dev_count);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    cudaEventElapsedTime(&old_time, start, stop);
    old_time /= ITERATIONS;

    cudaMemset(dev_count, 0, sizeof(int));

    cudaEventRecord(start);

    for (int i = 0; i < ITERATIONS; i++)
        operate_optimized<<<gridSize, blockSize>>>(dev_A, dev_count);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    cudaEventElapsedTime(&optimized_time, start, stop);
    optimized_time /= ITERATIONS;

    printf("k = %d\n\n", K);

    printf("CPU rezultat: %d\n", CPU_count);
    printf("Stari GPU rezultat: %d\n", old_count);
    printf("Optimizovani GPU rezultat: %d\n\n", optimized_count);

    if (CPU_count == old_count && CPU_count == optimized_count)
        printf("Rezultat je tacan!\n\n");
    else
        printf("Rezultat nije tacan!\n\n");

    printf("Prosecno vreme starog kernela: %.6f ms\n", old_time);
    printf("Prosecno vreme optimizovanog kernela: %.6f ms\n", optimized_time);
    printf("Ubrzanje: %.2fx\n", old_time / optimized_time);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(dev_A);
    cudaFree(dev_count);
    free(A);

    return 0;
}

__host__ int check_result(int *A)
{
    int count = 0;

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            if (A[i * N + j] == K)
                count++;
        }
    }

    return count;
}

__global__ void operate_old(int *A, int *count)
{
    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int tid_y = blockIdx.y * blockDim.y + threadIdx.y;

    for (int i = tid_x; i < N; i += gridDim.x * blockDim.x)
    {
        for (int j = tid_y; j < N; j += gridDim.y * blockDim.y)
        {
            if (A[i * N + j] == K)
                atomicAdd(count, 1);
        }
    }
}

__global__ void operate_optimized(int *A, int *count)
{
    __shared__ int block_count;

    if (threadIdx.x == 0 && threadIdx.y == 0)
        block_count = 0;

    __syncthreads();

    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int tid_y = blockIdx.y * blockDim.y + threadIdx.y;

    for (int i = tid_x; i < N; i += gridDim.x * blockDim.x)
    {
        for (int j = tid_y; j < N; j += gridDim.y * blockDim.y)
        {
            if (A[i * N + j] == K)
                atomicAdd(&block_count, 1);
        }
    }

    __syncthreads();

    if (threadIdx.x == 0 && threadIdx.y == 0)
        atomicAdd(count, block_count);
}

k = 5

CPU rezultat: 1677722
Stari GPU rezultat: 1677722
Optimizovani GPU rezultat: 1677722

Rezultat je tacan!

Prosecno vreme starog kernela: 1.164750 ms
Prosecno vreme optimizovanog kernela: 0.986100 ms
Ubrzanje: 1.18x



# 📝 Zadatak 7 — Maksimum elemenata dva vektora

Napisati CUDA program koji za data dva vektora `A` i `B` kreira treći vektor `C`.

Za svaki element važi:

$
C[i] = \max(A[i], B[i])
$

Dakle, za svaku poziciju `i` u vektor `C` upisuje se veća vrednost između odgovarajućih elemenata vektora `A` i `B`.

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti CPU i GPU rezultat.

Globalni indeks niti računa se kao:

```cpp
int tid = blockIdx.x * blockDim.x + threadIdx.x;
```

Za obradu vektora koristi se grid-stride petlja:

```cpp
for (int i = tid; i < N; i += gridDim.x * blockDim.x)
```

In [31]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 5120
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C);
__global__ void operate(int *A, int *B, int *C);
__host__ bool check_result(int *A, int *B, int *C);

__device__ __host__ int operation(int a, int b)
{
    return a > b ? a : b;
}

int main()
{
    int *A, *B, *C;

    A = (int *)malloc(sizeof(int) * N);
    B = (int *)malloc(sizeof(int) * N);
    C = (int *)malloc(sizeof(int) * N);

    for (int i = 0; i < N; i++)
    {
        A[i] = i * i;
        B[i] = N - (i + 1);
    }

    operate_on_GPU(A, B, C);

    printf("Prvih 20 elemenata:\n\n");

    for (int i = 0; i < 20; i++)
        printf("A[%d] = %d\tB[%d] = %d\tC[%d] = %d\n",
               i, A[i], i, B[i], i, C[i]);

    if (check_result(A, B, C))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat nije tacan!\n");

    free(C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *C)
{
    for (int i = 0; i < N; i++)
    {
        if (C[i] != operation(A[i], B[i]))
            return false;
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((N + blockSize.x - 1) / blockSize.x);

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    for (int i = tid; i < N; i += gridDim.x * blockDim.x)
        C[i] = operation(A[i], B[i]);
}

Prvih 20 elemenata:

A[0] = 0	B[0] = 5119	C[0] = 5119
A[1] = 1	B[1] = 5118	C[1] = 5118
A[2] = 4	B[2] = 5117	C[2] = 5117
A[3] = 9	B[3] = 5116	C[3] = 5116
A[4] = 16	B[4] = 5115	C[4] = 5115
A[5] = 25	B[5] = 5114	C[5] = 5114
A[6] = 36	B[6] = 5113	C[6] = 5113
A[7] = 49	B[7] = 5112	C[7] = 5112
A[8] = 64	B[8] = 5111	C[8] = 5111
A[9] = 81	B[9] = 5110	C[9] = 5110
A[10] = 100	B[10] = 5109	C[10] = 5109
A[11] = 121	B[11] = 5108	C[11] = 5108
A[12] = 144	B[12] = 5107	C[12] = 5107
A[13] = 169	B[13] = 5106	C[13] = 5106
A[14] = 196	B[14] = 5105	C[14] = 5105
A[15] = 225	B[15] = 5104	C[15] = 5104
A[16] = 256	B[16] = 5103	C[16] = 5103
A[17] = 289	B[17] = 5102	C[17] = 5102
A[18] = 324	B[18] = 5101	C[18] = 5101
A[19] = 361	B[19] = 5100	C[19] = 5100

Rezultat je tacan!



# 📝 Zadatak 8 — Maksimum elemenata dve matrice

Napisati CUDA program koji za date dve kvadratne matrice `A` i `B` kreira treću matricu `C`.

Za svaki element važi:

$
C[i][j] = \max(A[i][j], B[i][j])
$

Dakle, za svaku poziciju `[i][j]` u matricu `C` upisuje se veća vrednost između odgovarajućih elemenata matrica `A` i `B`.

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti CPU i GPU rezultat.

Za matricu koristimo dvodimenzionalnu organizaciju niti:

```cpp
int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
int tid_y = blockIdx.y * blockDim.y + threadIdx.y;
```

Za obradu matrice koriste se grid-stride petlje.

In [32]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 512
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C);
__global__ void operate(int *A, int *B, int *C);
__host__ bool check_result(int *A, int *B, int *C);

__device__ __host__ int operation(int a, int b)
{
    return a > b ? a : b;
}

int main()
{
    int *A, *B, *C;

    A = (int *)malloc(sizeof(int) * N * N);
    B = (int *)malloc(sizeof(int) * N * N);
    C = (int *)malloc(sizeof(int) * N * N);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            A[i * N + j] = i * N + j;
            B[i * N + j] = j * N + i;
        }
    }

    operate_on_GPU(A, B, C);

    printf("Prvih 8x8 elemenata matrice C:\n\n");

    for (int i = 0; i < 8; i++)
    {
        printf("|\t");

        for (int j = 0; j < 8; j++)
            printf("%d\t", C[i * N + j]);

        printf("|\n");
    }

    if (check_result(A, B, C))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat nije tacan!\n");

    free(C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *C)
{
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            int index = i * N + j;

            if (C[index] != operation(A[index], B[index]))
                return false;
        }
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridSize(
        (N + blockSize.x - 1) / blockSize.x,
        (N + blockSize.y - 1) / blockSize.y
    );

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C)
{
    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int tid_y = blockIdx.y * blockDim.y + threadIdx.y;

    for (int i = tid_x; i < N; i += gridDim.x * blockDim.x)
    {
        for (int j = tid_y; j < N; j += gridDim.y * blockDim.y)
        {
            int index = i * N + j;
            C[index] = operation(A[index], B[index]);
        }
    }
}

Prvih 8x8 elemenata matrice C:

|	0	512	1024	1536	2048	2560	3072	3584	|
|	512	513	1025	1537	2049	2561	3073	3585	|
|	1024	1025	1026	1538	2050	2562	3074	3586	|
|	1536	1537	1538	1539	2051	2563	3075	3587	|
|	2048	2049	2050	2051	2052	2564	3076	3588	|
|	2560	2561	2562	2563	2564	2565	3077	3589	|
|	3072	3073	3074	3075	3076	3077	3078	3590	|
|	3584	3585	3586	3587	3588	3589	3590	3591	|

Rezultat je tacan!



# 📝 Zadatak 9 — Oduzimanje vektora sa zamenom negativnih vrednosti

Napisati CUDA program koji računa izraz:

$
C = A - B
$

gde su `A` i `B` vektori.

Za svaki element najpre računamo:

$
C[i] = A[i] - B[i]
$

Ako je dobijena vrednost negativna, zamenjuje se nulom:

$
C[i] =
\begin{cases}
0, & \text{ako } A[i] - B[i] < 0 \\
A[i] - B[i], & \text{inače}
\end{cases}
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti CPU i GPU rezultat.
4. Analizirati ponašanje programa pri promeni broja blokova i broja niti po bloku.

Globalni indeks niti računa se kao:

```cpp
int tid = blockIdx.x * blockDim.x + threadIdx.x;
```

Za obradu vektora koristi se grid-stride petlja:

```cpp
for (int i = tid; i < N; i += gridDim.x * blockDim.x)
```

In [33]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 5120
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C);
__global__ void operate(int *A, int *B, int *C);
__host__ bool check_result(int *A, int *B, int *C);

__device__ __host__ int operation(int a, int b)
{
    int result = a - b;
    return result < 0 ? 0 : result;
}

int main()
{
    int *A, *B, *C;

    A = (int *)malloc(sizeof(int) * N);
    B = (int *)malloc(sizeof(int) * N);
    C = (int *)malloc(sizeof(int) * N);

    for (int i = 0; i < N; i++)
    {
        A[i] = i;
        B[i] = N - (i + 1);
    }

    operate_on_GPU(A, B, C);

    printf("Prvih 20 elemenata:\n\n");

    for (int i = 0; i < 20; i++)
        printf("A[%d] = %d\tB[%d] = %d\tC[%d] = %d\n",
               i, A[i], i, B[i], i, C[i]);

    if (check_result(A, B, C))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat nije tacan!\n");

    free(C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *C)
{
    for (int i = 0; i < N; i++)
    {
        if (C[i] != operation(A[i], B[i]))
            return false;
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((N + blockSize.x - 1) / blockSize.x);

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    for (int i = tid; i < N; i += gridDim.x * blockDim.x)
        C[i] = operation(A[i], B[i]);
}

Prvih 20 elemenata:

A[0] = 0	B[0] = 5119	C[0] = 0
A[1] = 1	B[1] = 5118	C[1] = 0
A[2] = 2	B[2] = 5117	C[2] = 0
A[3] = 3	B[3] = 5116	C[3] = 0
A[4] = 4	B[4] = 5115	C[4] = 0
A[5] = 5	B[5] = 5114	C[5] = 0
A[6] = 6	B[6] = 5113	C[6] = 0
A[7] = 7	B[7] = 5112	C[7] = 0
A[8] = 8	B[8] = 5111	C[8] = 0
A[9] = 9	B[9] = 5110	C[9] = 0
A[10] = 10	B[10] = 5109	C[10] = 0
A[11] = 11	B[11] = 5108	C[11] = 0
A[12] = 12	B[12] = 5107	C[12] = 0
A[13] = 13	B[13] = 5106	C[13] = 0
A[14] = 14	B[14] = 5105	C[14] = 0
A[15] = 15	B[15] = 5104	C[15] = 0
A[16] = 16	B[16] = 5103	C[16] = 0
A[17] = 17	B[17] = 5102	C[17] = 0
A[18] = 18	B[18] = 5101	C[18] = 0
A[19] = 19	B[19] = 5100	C[19] = 0

Rezultat je tacan!



# 📝 Zadatak 10 — Oduzimanje matrica sa zamenom negativnih vrednosti

Napisati CUDA program koji računa izraz:

$
C = A - B
$

gde su `A` i `B` kvadratne matrice.

Za svaki element najpre računamo:

$
C[i][j] = A[i][j] - B[i][j]
$

Ako je dobijena vrednost negativna, zamenjuje se nulom:

$
C[i][j] =
\begin{cases}
0, & \text{ako } A[i][j] - B[i][j] < 0 \\
A[i][j] - B[i][j], & \text{inače}
\end{cases}
$

Potrebno je:

1. Izračunati rezultat paralelno korišćenjem CUDA kernela.
2. Izračunati isti rezultat sekvencijalno na CPU-u.
3. Uporediti CPU i GPU rezultat.
4. Analizirati ponašanje programa pri promeni broja blokova i broja niti po bloku.

Za matricu koristimo dvodimenzionalnu organizaciju niti:

```cpp
int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
int tid_y = blockIdx.y * blockDim.y + threadIdx.y;
```

Za obradu matrice koriste se grid-stride petlje.

In [34]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N 512
#define BLOCK_SIZE 32

__host__ void operate_on_GPU(int *A, int *B, int *C);
__global__ void operate(int *A, int *B, int *C);
__host__ bool check_result(int *A, int *B, int *C);

__device__ __host__ int operation(int a, int b)
{
    int result = a - b;
    return result < 0 ? 0 : result;
}

int main()
{
    int *A, *B, *C;

    A = (int *)malloc(sizeof(int) * N * N);
    B = (int *)malloc(sizeof(int) * N * N);
    C = (int *)malloc(sizeof(int) * N * N);

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            A[i * N + j] = i * N + j;
            B[i * N + j] = j * N + i;
        }
    }

    operate_on_GPU(A, B, C);

    printf("Prvih 8x8 elemenata matrice C:\n\n");

    for (int i = 0; i < 8; i++)
    {
        printf("|\t");

        for (int j = 0; j < 8; j++)
            printf("%d\t", C[i * N + j]);

        printf("|\n");
    }

    if (check_result(A, B, C))
        printf("\nRezultat je tacan!\n");
    else
        printf("\nRezultat nije tacan!\n");

    free(C);
    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, int *B, int *C)
{
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            int index = i * N + j;

            if (C[index] != operation(A[index], B[index]))
                return false;
        }
    }

    return true;
}

__host__ void operate_on_GPU(int *A, int *B, int *C)
{
    int *dev_A, *dev_B, *dev_C;
    size_t size = sizeof(int) * N * N;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_C, size);

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridSize(
        (N + blockSize.x - 1) / blockSize.x,
        (N + blockSize.y - 1) / blockSize.y
    );

    operate<<<gridSize, blockSize>>>(dev_A, dev_B, dev_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);
}

__global__ void operate(int *A, int *B, int *C)
{
    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int tid_y = blockIdx.y * blockDim.y + threadIdx.y;

    for (int i = tid_x; i < N; i += gridDim.x * blockDim.x)
    {
        for (int j = tid_y; j < N; j += gridDim.y * blockDim.y)
        {
            int index = i * N + j;
            C[index] = operation(A[index], B[index]);
        }
    }
}

Prvih 8x8 elemenata matrice C:

|	0	0	0	0	0	0	0	0	|
|	511	0	0	0	0	0	0	0	|
|	1022	511	0	0	0	0	0	0	|
|	1533	1022	511	0	0	0	0	0	|
|	2044	1533	1022	511	0	0	0	0	|
|	2555	2044	1533	1022	511	0	0	0	|
|	3066	2555	2044	1533	1022	511	0	0	|
|	3577	3066	2555	2044	1533	1022	511	0	|

Rezultat je tacan!



# 📝 Zadatak 11 — Računanje niza B korišćenjem shared memorije

Za dati niz `A[n+2]` potrebno je izračunati niz `B[n]` prema izrazu:

$
B[i] = \frac{3A[i] + 10A[i+1] + 7A[i+2]}{20}
$

Za svaki element `B[i]` koriste se tri uzastopna elementa niza `A`:

```text
B[0] koristi A[0], A[1], A[2]
B[1] koristi A[1], A[2], A[3]
B[2] koristi A[2], A[3], A[4]
...
```

Pošto se isti elementi niza `A` koriste više puta, koristi se **shared memorija** kako bi se smanjio broj pristupa globalnoj memoriji.

Svaka nit jednog bloka učitava jedan element niza `A` u shared memoriju:

```cpp
sh[threadIdx.x] = A[tid_x + 1];
```

Nakon toga se niti sinhronizuju:

```cpp
__syncthreads();
```

Unutrašnje niti koriste samo shared memoriju:

```cpp
B[tid_x] =
    (3 * sh[threadIdx.x - 1] +
     10 * sh[threadIdx.x] +
     7 * sh[threadIdx.x + 1]) / 20.f;
```

Prva nit u bloku nema element `sh[threadIdx.x - 1]`, pa prvi element uzima iz globalne memorije.

Poslednja nit u bloku nema element `sh[threadIdx.x + 1]`, pa poslednji potreban element uzima iz globalne memorije.

Na ovaj način se značajno smanjuje broj pristupa globalnoj memoriji.

Globalni indeks niti računa se kao:

```cpp
int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
```

Program omogućava rad za nizove proizvoljne veličine.

In [35]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 32

__host__ void operate(int *A, float *B, int n);
__host__ bool check_result(int *A, float *B, int n);
__global__ void compute_B(int *A, float *B, int n);

int main()
{
    srand(time(NULL));

    int sizeA;

    printf("Unesite velicinu niza A: ");
    scanf("%d", &sizeA);

    if (sizeA < 3)
    {
        printf("Niz A mora imati najmanje 3 elementa.\n");
        return 1;
    }

    int n = sizeA - 2;

    int *A = (int *)malloc(sizeof(int) * sizeA);
    float *B = (float *)malloc(sizeof(float) * n);

    for (int i = 0; i < sizeA; i++)
        A[i] = rand() % 100;

    operate(A, B, n);

    int limit = n < 20 ? n : 20;

    printf("\nNiz A:\n");

    for (int i = 0; i < limit + 2; i++)
        printf("A[%d] = %d\n", i, A[i]);

    printf("\nNiz B:\n");

    for (int i = 0; i < limit; i++)
        printf("B[%d] = %.2f\n", i, B[i]);

    if (check_result(A, B, n))
        printf("\nCorrect!\n");
    else
        printf("\nFalse!\n");

    free(B);
    free(A);

    return 0;
}

__host__ bool check_result(int *A, float *B, int n)
{
    for (int i = 0; i < n; i++)
    {
        float result = (3 * A[i] + 10 * A[i + 1] + 7 * A[i + 2]) / 20.f;

        if (fabsf(B[i] - result) > 0.0001f)
            return false;
    }

    return true;
}

__host__ void operate(int *A, float *B, int n)
{
    int *dev_A;
    float *dev_B;

    size_t count_A = sizeof(int) * (n + 2);
    size_t count_B = sizeof(float) * n;

    cudaMalloc((void **)&dev_A, count_A);
    cudaMalloc((void **)&dev_B, count_B);

    cudaMemcpy(dev_A, A, count_A, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((n + blockSize.x - 1) / blockSize.x);

    compute_B<<<gridSize, blockSize>>>(dev_A, dev_B, n);

    cudaDeviceSynchronize();

    cudaMemcpy(B, dev_B, count_B, cudaMemcpyDeviceToHost);

    cudaFree(dev_B);
    cudaFree(dev_A);
}

__global__ void compute_B(int *A, float *B, int n)
{
    __shared__ int sh[BLOCK_SIZE];

    int tid_x = blockIdx.x * blockDim.x + threadIdx.x;
    int load_index = tid_x + 1;

    if (load_index < n + 2)
        sh[threadIdx.x] = A[load_index];
    else
        sh[threadIdx.x] = 0;

    __syncthreads();

    if (tid_x < n)
    {
        if (threadIdx.x == 0)
        {
            B[tid_x] = (3 * A[tid_x] + 10 * sh[threadIdx.x] + 7 * sh[threadIdx.x + 1]) / 20.f;
        }
        else if (threadIdx.x == blockDim.x - 1)
        {
            B[tid_x] = (3 * sh[threadIdx.x - 1] + 10 * sh[threadIdx.x] + 7 * A[tid_x + 2]) / 20.f;
        }
        else
        {
            B[tid_x] = (3 * sh[threadIdx.x - 1] + 10 * sh[threadIdx.x] + 7 * sh[threadIdx.x + 1]) / 20.f;
        }
    }
}

Unesite velicinu niza A: Niz A mora imati najmanje 3 elementa.



# 📝 Zadatak 12 — Minimum i maksimum svake kolone matrice

Za matricu `M` dimenzija `n × m` potrebno je pronaći minimalni i maksimalni element svake kolone.

Za svaku kolonu `j` određujemo:

$
MIN[j] = \min_i M[i][j]
$

$
MAX[j] = \max_i M[i][j]
$

Koristi se jednostavna organizacija:

```text
1 CUDA blok = 1 kolona matrice
```

Indeks kolone određujemo pomoću:

```cpp
int col = blockIdx.x;
```

Sve niti jednog bloka zajednički obrađuju tu kolonu.

Svaka nit počinje od reda koji odgovara njenom indeksu:

```cpp
int tid = threadIdx.x;
```

i prolazi kroz redove:

```cpp
for (int row = tid; row < n; row += blockDim.x)
```

Na primer, za 32 niti:

```text
T0 → redovi 0, 32, 64, ...
T1 → redovi 1, 33, 65, ...
T2 → redovi 2, 34, 66, ...
...
T31 → redovi 31, 63, 95, ...
```

Svaka nit pronalazi svoj lokalni minimum i maksimum:

```cpp
local_min
local_max
```

Nakon toga se lokalni rezultati smeštaju u shared memoriju:

```cpp
sh_min[tid] = local_min;
sh_max[tid] = local_max;
```

Zatim se radi paralelna redukcija:

```text
32 → 16 → 8 → 4 → 2 → 1
```

U svakom koraku za minimum se zadržava manja vrednost, a za maksimum veća vrednost.

Na kraju nit `0` iz svakog bloka upisuje konačni rezultat svoje kolone:

```cpp
res_min[col] = sh_min[0];
res_max[col] = sh_max[0];
```

Mentalno:

```text
KOLONA
↓
jedan CUDA blok
↓
32 niti dele redove
↓
svaka nit pronađe svoj min/max
↓
shared memory
↓
32 → 16 → 8 → 4 → 2 → 1
↓
MIN i MAX kolone
```

In [36]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <limits.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 32

__host__ void operate(int *M, int *res_min, int *res_max, int n, int m);
__host__ bool check_result(int *M, int *res_min, int *res_max, int n, int m);
__global__ void find_min_max(int *M, int n, int m, int *res_min, int *res_max);

int main()
{
    srand(time(NULL));

    int n = 128, m = 256;

    int *M = (int *)malloc(sizeof(int) * n * m);
    int *res_min = (int *)malloc(sizeof(int) * m);
    int *res_max = (int *)malloc(sizeof(int) * m);

    for (int i = 0; i < n * m; i++)
        M[i] = rand() % 100;

    operate(M, res_min, res_max, n, m);

    if (n <= 10 && m <= 10)
    {
        printf("\nMatrica M:\n\n");

        for (int i = 0; i < n; i++)
        {
            for (int j = 0; j < m; j++)
                printf("%d\t", M[i * m + j]);

            printf("\n");
        }
    }

    printf("\nMinimum i maksimum svake kolone:\n\n");

    int limit = m < 10 ? m : 10;

    for (int j = 0; j < limit; j++)
        printf("Kolona %d: min = %d, max = %d\n",
               j, res_min[j], res_max[j]);

    if (check_result(M, res_min, res_max, n, m))
        printf("\nCorrect!\n");
    else
        printf("\nFalse!\n");

    free(res_max);
    free(res_min);
    free(M);

    return 0;
}

__host__ void operate(int *M, int *res_min, int *res_max, int n, int m)
{
    int *dev_M;
    int *dev_min;
    int *dev_max;

    size_t matrix_size = sizeof(int) * n * m;
    size_t result_size = sizeof(int) * m;

    cudaMalloc((void **)&dev_M, matrix_size);
    cudaMalloc((void **)&dev_min, result_size);
    cudaMalloc((void **)&dev_max, result_size);

    cudaMemcpy(dev_M, M, matrix_size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize(m);

    find_min_max<<<gridSize, blockSize>>>(
        dev_M,
        n,
        m,
        dev_min,
        dev_max
    );

    cudaDeviceSynchronize();

    cudaMemcpy(res_min, dev_min, result_size, cudaMemcpyDeviceToHost);
    cudaMemcpy(res_max, dev_max, result_size, cudaMemcpyDeviceToHost);

    cudaFree(dev_max);
    cudaFree(dev_min);
    cudaFree(dev_M);
}

__host__ bool check_result(int *M, int *res_min, int *res_max, int n, int m)
{
    for (int j = 0; j < m; j++)
    {
        int min_value = M[j];
        int max_value = M[j];

        for (int i = 1; i < n; i++)
        {
            int value = M[i * m + j];

            if (value < min_value)
                min_value = value;

            if (value > max_value)
                max_value = value;
        }

        if (res_min[j] != min_value || res_max[j] != max_value)
            return false;
    }

    return true;
}

__global__ void find_min_max(int *M, int n, int m,
                             int *res_min, int *res_max)
{
    __shared__ int sh_min[BLOCK_SIZE];
    __shared__ int sh_max[BLOCK_SIZE];

    int col = blockIdx.x;
    int tid = threadIdx.x;

    int local_min = INT_MAX;
    int local_max = INT_MIN;

    for (int row = tid; row < n; row += blockDim.x)
    {
        int value = M[row * m + col];

        if (value < local_min)
            local_min = value;

        if (value > local_max)
            local_max = value;
    }

    sh_min[tid] = local_min;
    sh_max[tid] = local_max;

    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride /= 2)
    {
        if (tid < stride)
        {
            if (sh_min[tid + stride] < sh_min[tid])
                sh_min[tid] = sh_min[tid + stride];

            if (sh_max[tid + stride] > sh_max[tid])
                sh_max[tid] = sh_max[tid + stride];
        }

        __syncthreads();
    }

    if (tid == 0)
    {
        res_min[col] = sh_min[0];
        res_max[col] = sh_max[0];
    }
}


Minimum i maksimum svake kolone:

Kolona 0: min = 0, max = 99
Kolona 1: min = 2, max = 99
Kolona 2: min = 0, max = 99
Kolona 3: min = 0, max = 97
Kolona 4: min = 0, max = 99
Kolona 5: min = 0, max = 99
Kolona 6: min = 0, max = 99
Kolona 7: min = 1, max = 99
Kolona 8: min = 0, max = 99
Kolona 9: min = 0, max = 99

Correct!



# 📝 Zadatak 13 — Suma elemenata svake vrste matrice

Za kvadratnu matricu `A` dimenzija `n × n` potrebno je kreirati vektor `B` dužine `n`.

Svaki element vektora `B` predstavlja sumu elemenata odgovarajuće vrste matrice:

$
B[i] = \sum_{j=0}^{n-1} A[i][j]
$

Koristi se organizacija:

```text
1 CUDA blok = 1 vrsta matrice
```

Indeks vrste određujemo pomoću:

```cpp
int row = blockIdx.x;
```

Sve niti jednog bloka zajednički sabiraju elemente te vrste.

Svaka nit počinje od kolone koja odgovara njenom indeksu:

```cpp
int tid = threadIdx.x;
```

i prolazi kroz kolone:

```cpp
for (int col = tid; col < n; col += blockDim.x)
```

Na primer, za 32 niti:

```text
T0  → kolone 0, 32, 64, ...
T1  → kolone 1, 33, 65, ...
T2  → kolone 2, 34, 66, ...
...
T31 → kolone 31, 63, 95, ...
```

Svaka nit izračunava svoju parcijalnu sumu:

```cpp
local_sum
```

Parcijalne sume se zatim smeštaju u shared memoriju:

```cpp
sh_sum[tid] = local_sum;
```

Nakon toga se vrši paralelna redukcija:

```text
32 → 16 → 8 → 4 → 2 → 1
```

U svakom koraku sabiraju se dve parcijalne sume.

Na kraju nit `0` iz svakog bloka upisuje konačnu sumu odgovarajuće vrste:

```cpp
B[row] = sh_sum[0];
```

Mentalno:

```text
VRSTA MATRICE
      ↓
1 CUDA blok
      ↓
32 niti dele elemente vrste
      ↓
svaka nit računa svoju parcijalnu sumu
      ↓
shared memory
      ↓
32 → 16 → 8 → 4 → 2 → 1
      ↓
B[row] = suma cele vrste
```

In [37]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 32

__host__ void operate(int *A, int *B, int n);
__host__ bool check_result(int *A, int *B, int n);
__global__ void calculate_sum(int *A, int *B, int n);

int main()
{
    srand(time(NULL));

    int n = 128;

    int *A = (int *)malloc(sizeof(int) * n * n);
    int *B = (int *)malloc(sizeof(int) * n);

    for (int i = 0; i < n * n; i++)
        A[i] = rand() % 10;

    operate(A, B, n);

    if (n <= 10)
    {
        printf("\nMatrica A:\n\n");

        for (int i = 0; i < n; i++)
        {
            for (int j = 0; j < n; j++)
                printf("%d\t", A[i * n + j]);

            printf("\n");
        }
    }

    printf("\nVektor B:\n\n");

    int limit = n < 20 ? n : 20;

    for (int i = 0; i < limit; i++)
        printf("B[%d] = %d\n", i, B[i]);

    if (check_result(A, B, n))
        printf("\nCorrect!\n");
    else
        printf("\nFalse!\n");

    free(B);
    free(A);

    return 0;
}

__host__ void operate(int *A, int *B, int n)
{
    int *dev_A;
    int *dev_B;

    size_t matrix_size = sizeof(int) * n * n;
    size_t vector_size = sizeof(int) * n;

    cudaMalloc((void **)&dev_A, matrix_size);
    cudaMalloc((void **)&dev_B, vector_size);

    cudaMemcpy(dev_A, A, matrix_size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize(n);

    calculate_sum<<<gridSize, blockSize>>>(dev_A, dev_B, n);

    cudaDeviceSynchronize();

    cudaMemcpy(B, dev_B, vector_size, cudaMemcpyDeviceToHost);

    cudaFree(dev_B);
    cudaFree(dev_A);
}

__host__ bool check_result(int *A, int *B, int n)
{
    for (int i = 0; i < n; i++)
    {
        int sum = 0;

        for (int j = 0; j < n; j++)
            sum += A[i * n + j];

        if (B[i] != sum)
            return false;
    }

    return true;
}

__global__ void calculate_sum(int *A, int *B, int n)
{
    __shared__ int sh_sum[BLOCK_SIZE];

    int row = blockIdx.x;
    int tid = threadIdx.x;

    int local_sum = 0;

    for (int col = tid; col < n; col += blockDim.x)
        local_sum += A[row * n + col];

    sh_sum[tid] = local_sum;

    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride /= 2)
    {
        if (tid < stride)
            sh_sum[tid] += sh_sum[tid + stride];

        __syncthreads();
    }

    if (tid == 0)
        B[row] = sh_sum[0];
}


Vektor B:

B[0] = 580
B[1] = 553
B[2] = 574
B[3] = 557
B[4] = 604
B[5] = 628
B[6] = 591
B[7] = 621
B[8] = 578
B[9] = 589
B[10] = 562
B[11] = 586
B[12] = 536
B[13] = 585
B[14] = 621
B[15] = 598
B[16] = 519
B[17] = 612
B[18] = 531
B[19] = 655

Correct!



# 📝 Zadatak 14 — Skalarni proizvod dva niza

Za dva niza `A` i `B` dužine `n` potrebno je izračunati njihov skalarni proizvod:

$
result = \sum_{i=0}^{n-1} A[i] \cdot B[i]
$

Svaka CUDA nit obrađuje jedan ili više elemenata nizova.

Globalni indeks niti računa se kao:

```cpp
int index = blockIdx.x * blockDim.x + threadIdx.x;
```

Nit zatim prolazi kroz elemente pomoću grid-stride petlje:

```cpp
for (int i = index; i < n; i += gridDim.x * blockDim.x)
```

i računa svoju parcijalnu sumu:

```cpp
local_sum += A[i] * B[i];
```

Parcijalna suma svake niti smešta se u shared memoriju:

```cpp
sh_sum[tid] = local_sum;
```

Nakon toga se radi paralelna redukcija:

```text
32 → 16 → 8 → 4 → 2 → 1
```

U svakom koraku sabiraju se dve parcijalne sume.

Na kraju u svakom bloku ostaje jedna suma:

```cpp
sh_sum[0]
```

Samo nit `0` iz svakog bloka dodaje parcijalni rezultat bloka u konačni rezultat:

```cpp
atomicAdd(result, sh_sum[0]);
```

Mentalno:

```text
A[i] × B[i]
     ↓
svaka nit računa svoj deo
     ↓
local_sum
     ↓
shared memory
     ↓
32 → 16 → 8 → 4 → 2 → 1
     ↓
suma jednog bloka
     ↓
atomicAdd
     ↓
konačni skalarni proizvod
```

In [38]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 32

__host__ void operate(int *A, int *B, int *result, int n);
__host__ bool check_result(int *A, int *B, int result, int n);
__global__ void scalar_product(int *A, int *B, int *result, int n);

int main()
{
    srand(time(NULL));

    int n = 4096;

    int *A = (int *)malloc(sizeof(int) * n);
    int *B = (int *)malloc(sizeof(int) * n);

    for (int i = 0; i < n; i++)
    {
        A[i] = rand() % 10;
        B[i] = rand() % 10;
    }

    int result = 0;

    operate(A, B, &result, n);

    printf("Skalarni proizvod = %d\n", result);

    if (check_result(A, B, result, n))
        printf("Correct!\n");
    else
        printf("False!\n");

    free(B);
    free(A);

    return 0;
}

__host__ void operate(int *A, int *B, int *result, int n)
{
    int *dev_A;
    int *dev_B;
    int *dev_result;

    size_t size = sizeof(int) * n;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_B, size);
    cudaMalloc((void **)&dev_result, sizeof(int));

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    cudaMemset(dev_result, 0, sizeof(int));

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((n + BLOCK_SIZE - 1) / BLOCK_SIZE);

    scalar_product<<<gridSize, blockSize>>>(
        dev_A,
        dev_B,
        dev_result,
        n
    );

    cudaDeviceSynchronize();

    cudaMemcpy(
        result,
        dev_result,
        sizeof(int),
        cudaMemcpyDeviceToHost
    );

    cudaFree(dev_result);
    cudaFree(dev_B);
    cudaFree(dev_A);
}

__host__ bool check_result(int *A, int *B, int result, int n)
{
    int sum = 0;

    for (int i = 0; i < n; i++)
        sum += A[i] * B[i];

    return sum == result;
}

__global__ void scalar_product(int *A, int *B, int *result, int n)
{
    __shared__ int sh_sum[BLOCK_SIZE];

    int tid = threadIdx.x;
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    int local_sum = 0;

    for (int i = index; i < n; i += gridDim.x * blockDim.x)
        local_sum += A[i] * B[i];

    sh_sum[tid] = local_sum;

    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride /= 2)
    {
        if (tid < stride)
            sh_sum[tid] += sh_sum[tid + stride];

        __syncthreads();
    }

    if (tid == 0)
        atomicAdd(result, sh_sum[0]);
}

Skalarni proizvod = 82165
Correct!



# 📝 Zadatak 15 — Zamena elemenata većih od prosečne vrednosti

Za matricu `A` dimenzija `n × m` potrebno je izračunati prosečnu vrednost svih elemenata:

$
avg = \frac{\sum A[i][j]}{n \cdot m}
$

Nakon toga svaki element koji je veći od prosečne vrednosti zamenjuje se brojem `-1`:

$$
A[i][j] =
\begin{cases}
-1, & A[i][j] > avg \\
A[i][j], & A[i][j] \leq avg
\end{cases}
$$

Rešenje se izvršava u dva koraka.

## 1. Izračunavanje sume elemenata

Svaka nit sabira jedan ili više elemenata matrice:

```cpp
local_sum += A[i];
```

Parcijalne sume smeštaju se u shared memoriju:

```cpp
sh_sum[tid] = local_sum;
```

Zatim se vrši redukcija:

```text
32 → 16 → 8 → 4 → 2 → 1
```

Na kraju svaki blok dobija svoju parcijalnu sumu, koju nit `0` dodaje ukupnoj sumi:

```cpp
atomicAdd(sum, sh_sum[0]);
```

Prosečna vrednost se zatim dobija kao:

```cpp
average = (float)sum / (n * m);
```

## 2. Zamena elemenata

Drugi kernel prolazi kroz matricu i proverava svaki element:

```cpp
if (A[i] > average)
    A[i] = -1;
```

Mentalno:

```text
MATRICA A
    ↓
paralelno sabiranje
    ↓
shared memory + reduction
    ↓
SUMA
    ↓
PROSEK
    ↓
ponovni paralelni prolazak kroz A
    ↓
A[i] > prosek ?
   ↓       ↓
  DA       NE
  -1    ostaje isto
```

In [39]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 32

__host__ float operate(int *A, int n, int m);
__host__ bool check_result(int *original, int *A, float average, int n, int m);

__global__ void calculate_sum(int *A, int *sum, int total);
__global__ void replace_greater(int *A, float average, int total);

int main()
{
    srand(time(NULL));

    int n = 128;
    int m = 256;

    int total = n * m;

    int *A = (int *)malloc(sizeof(int) * total);
    int *original = (int *)malloc(sizeof(int) * total);

    for (int i = 0; i < total; i++)
    {
        A[i] = rand() % 100;
        original[i] = A[i];
    }

    float average = operate(A, n, m);

    printf("Prosecna vrednost = %.2f\n\n", average);

    printf("Prvih 20 elemenata nakon izmene:\n\n");

    for (int i = 0; i < 20; i++)
        printf("A[%d] = %d\n", i, A[i]);

    if (check_result(original, A, average, n, m))
        printf("\nCorrect!\n");
    else
        printf("\nFalse!\n");

    free(original);
    free(A);

    return 0;
}

__host__ float operate(int *A, int n, int m)
{
    int *dev_A;
    int *dev_sum;

    int total = n * m;

    size_t size = sizeof(int) * total;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_sum, sizeof(int));

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);

    cudaMemset(dev_sum, 0, sizeof(int));

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((total + BLOCK_SIZE - 1) / BLOCK_SIZE);

    calculate_sum<<<gridSize, blockSize>>>(
        dev_A,
        dev_sum,
        total
    );

    cudaDeviceSynchronize();

    int sum;

    cudaMemcpy(
        &sum,
        dev_sum,
        sizeof(int),
        cudaMemcpyDeviceToHost
    );

    float average = (float)sum / total;

    replace_greater<<<gridSize, blockSize>>>(
        dev_A,
        average,
        total
    );

    cudaDeviceSynchronize();

    cudaMemcpy(
        A,
        dev_A,
        size,
        cudaMemcpyDeviceToHost
    );

    cudaFree(dev_sum);
    cudaFree(dev_A);

    return average;
}

__host__ bool check_result(int *original, int *A, float average, int n, int m)
{
    int total = n * m;

    for (int i = 0; i < total; i++)
    {
        int expected;

        if (original[i] > average)
            expected = -1;
        else
            expected = original[i];

        if (A[i] != expected)
            return false;
    }

    return true;
}

__global__ void calculate_sum(int *A, int *sum, int total)
{
    __shared__ int sh_sum[BLOCK_SIZE];

    int tid = threadIdx.x;
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    int local_sum = 0;

    for (int i = index; i < total; i += gridDim.x * blockDim.x)
    {
        local_sum += A[i];
    }

    sh_sum[tid] = local_sum;

    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride /= 2)
    {
        if (tid < stride)
            sh_sum[tid] += sh_sum[tid + stride];

        __syncthreads();
    }

    if (tid == 0)
        atomicAdd(sum, sh_sum[0]);
}

__global__ void replace_greater(int *A, float average, int total)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    for (int i = index; i < total; i += gridDim.x * blockDim.x)
    {
        if (A[i] > average)
            A[i] = -1;
    }
}

Prosecna vrednost = 49.42

Prvih 20 elemenata nakon izmene:

A[0] = 20
A[1] = 10
A[2] = 46
A[3] = -1
A[4] = -1
A[5] = -1
A[6] = 34
A[7] = 29
A[8] = 44
A[9] = -1
A[10] = -1
A[11] = -1
A[12] = -1
A[13] = 36
A[14] = 41
A[15] = 44
A[16] = -1
A[17] = 41
A[18] = 26
A[19] = -1

Correct!



# 📝 Zadatak 16 — Zamena negativnih elemenata najmanjim pozitivnim

Za niz `A` potrebno je pronaći najmanji pozitivan element:

$
min\_positive = \min \{ A[i] \mid A[i] > 0 \}
$

Nakon toga svaki negativni element niza menjamo tom vrednošću:

$
A[i] =
\begin{cases}
min\_positive, & A[i] < 0 \\
A[i], & A[i] \geq 0
\end{cases}
$

Rešenje se izvršava u dva koraka.

## 1. Pronalaženje najmanjeg pozitivnog elementa

Svaka nit prolazi kroz svoj deo niza i pronalazi svoj lokalni minimum:

```cpp
local_min
```

U obzir se uzimaju samo pozitivni elementi:

```cpp
if (A[i] > 0 && A[i] < local_min)
    local_min = A[i];
```

Lokalni minimumi se smeštaju u shared memoriju:

```cpp
sh_min[tid] = local_min;
```

Zatim se radi paralelna redukcija:

```text
32 → 16 → 8 → 4 → 2 → 1
```

Na kraju svaki blok dobija jedan parcijalni minimum, a nit `0` ga poredi sa globalnim minimumom.

## 2. Zamena negativnih elemenata

Drugi kernel prolazi kroz niz:

```cpp
if (A[i] < 0)
    A[i] = min_positive;
```

Mentalno:

```text
NIZ A
  ↓
tražimo samo pozitivne brojeve
  ↓
lokalni minimumi niti
  ↓
shared memory
  ↓
32 → 16 → 8 → 4 → 2 → 1
  ↓
najmanji pozitivan element
  ↓
drugi prolazak kroz A
  ↓
negativan?
  ↓
zameni sa min_positive
```

In [40]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <limits.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 32

__host__ int operate(int *A, int n);
__host__ bool check_result(int *original, int *A, int min_positive, int n);

__global__ void find_min_positive(int *A, int *min_positive, int n);
__global__ void replace_negative(int *A, int min_positive, int n);

int main()
{
    srand(time(NULL));

    int n = 4096;

    int *A = (int *)malloc(sizeof(int) * n);
    int *original = (int *)malloc(sizeof(int) * n);

    for (int i = 0; i < n; i++)
    {
        A[i] = rand() % 201 - 100;
        original[i] = A[i];
    }

    int min_positive = operate(A, n);

    printf("Najmanji pozitivan element = %d\n\n", min_positive);

    printf("Prvih 20 elemenata nakon izmene:\n\n");

    for (int i = 0; i < 20; i++)
        printf("A[%d] = %d\n", i, A[i]);

    if (check_result(original, A, min_positive, n))
        printf("\nCorrect!\n");
    else
        printf("\nFalse!\n");

    free(original);
    free(A);

    return 0;
}

__host__ int operate(int *A, int n)
{
    int *dev_A;
    int *dev_min;

    size_t size = sizeof(int) * n;

    cudaMalloc((void **)&dev_A, size);
    cudaMalloc((void **)&dev_min, sizeof(int));

    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);

    int initial_min = INT_MAX;

    cudaMemcpy(
        dev_min,
        &initial_min,
        sizeof(int),
        cudaMemcpyHostToDevice
    );

    dim3 blockSize(BLOCK_SIZE);
    dim3 gridSize((n + BLOCK_SIZE - 1) / BLOCK_SIZE);

    find_min_positive<<<gridSize, blockSize>>>(
        dev_A,
        dev_min,
        n
    );

    cudaDeviceSynchronize();

    int min_positive;

    cudaMemcpy(
        &min_positive,
        dev_min,
        sizeof(int),
        cudaMemcpyDeviceToHost
    );

    if (min_positive != INT_MAX)
    {
        replace_negative<<<gridSize, blockSize>>>(
            dev_A,
            min_positive,
            n
        );

        cudaDeviceSynchronize();
    }

    cudaMemcpy(
        A,
        dev_A,
        size,
        cudaMemcpyDeviceToHost
    );

    cudaFree(dev_min);
    cudaFree(dev_A);

    return min_positive;
}

__host__ bool check_result(int *original, int *A,
                           int min_positive, int n)
{
    for (int i = 0; i < n; i++)
    {
        int expected = original[i];

        if (original[i] < 0)
            expected = min_positive;

        if (A[i] != expected)
            return false;
    }

    return true;
}

__global__ void find_min_positive(int *A, int *min_positive, int n)
{
    __shared__ int sh_min[BLOCK_SIZE];

    int tid = threadIdx.x;
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    int local_min = INT_MAX;

    for (int i = index;
         i < n;
         i += gridDim.x * blockDim.x)
    {
        if (A[i] > 0 && A[i] < local_min)
            local_min = A[i];
    }

    sh_min[tid] = local_min;

    __syncthreads();

    for (int stride = blockDim.x / 2;
         stride > 0;
         stride /= 2)
    {
        if (tid < stride)
        {
            if (sh_min[tid + stride] < sh_min[tid])
                sh_min[tid] = sh_min[tid + stride];
        }

        __syncthreads();
    }

    if (tid == 0)
        atomicMin(min_positive, sh_min[0]);
}

__global__ void replace_negative(int *A, int min_positive, int n)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    for (int i = index;
         i < n;
         i += gridDim.x * blockDim.x)
    {
        if (A[i] < 0)
            A[i] = min_positive;
    }
}

Najmanji pozitivan element = 1

Prvih 20 elemenata nakon izmene:

A[0] = 33
A[1] = 16
A[2] = 25
A[3] = 1
A[4] = 1
A[5] = 1
A[6] = 19
A[7] = 70
A[8] = 1
A[9] = 38
A[10] = 91
A[11] = 1
A[12] = 9
A[13] = 1
A[14] = 1
A[15] = 97
A[16] = 44
A[17] = 14
A[18] = 1
A[19] = 1

Correct!

